In [ ]:
"""Design-matrix formula resolution for time-varying parameters."""

import ast
import operator
from typing import Dict, Mapping, Sequence
import numpy as np


_BIN_OPS = {
    ast.Add: operator.add,
    ast.Sub: operator.sub,
    ast.Mult: operator.mul,
    ast.Div: operator.truediv,
    ast.Pow: operator.pow,
}
_UNARY_OPS = {
    ast.UAdd: operator.pos,
    ast.USub: operator.neg,
}


class DesignMatrix:
    """Translate string formulas into operations on parameter/covariate arrays.

    Each formula has the form ``"target = <expression>"``, where the
    expression may reference sampled parameters, covariates, and numeric
    literals combined with + - * / ** and parentheses. Example:

        "v = v_0 + b_v * covariate"

    Parameters
    ----------
    formulas : sequence of str
        Formulas of the form ``"target = expression"``. Each `target`
        becomes a key in the resolved output.

    Notes
    -----
    Names on the right-hand side are looked up at evaluation time in the
    merged dict of `params` and `covariates`. All arrays are expected to
    share shape ``(num_steps,)`` (scalars broadcast).
    """

    def __init__(self, formulas: Sequence[str]):
        self.targets: list[str] = []
        self.trees: list[ast.Expression] = []
        self.required_names: set[str] = set()

        for formula in formulas:
            if "=" not in formula:
                raise ValueError(f"Formula must contain '=': {formula!r}")
            target, expr = formula.split("=", 1)
            target = target.strip()
            if not target.isidentifier():
                raise ValueError(f"Invalid target name {target!r} in formula {formula!r}")

            tree = ast.parse(expr.strip(), mode="eval")
            self._validate(tree, formula)

            self.targets.append(target)
            self.trees.append(tree)
            self.required_names |= self._collect_names(tree)

    def resolve(
        self,
        params: Mapping[str, np.ndarray],
        covariates: Mapping[str, np.ndarray] | None = None,
    ) -> Dict[str, np.ndarray]:
        """Evaluate all formulas into a dict of target arrays.

        Parameters
        ----------
        params     : dict of np.ndarray
            Sampled parameters, each of shape (num_steps,).
        covariates : dict of np.ndarray or None, optional, default: None
            Covariate arrays of shape (num_steps,).

        Returns
        -------
        resolved : dict of np.ndarray - one entry per formula target,
            each of shape (num_steps,)

        Raises
        ------
        KeyError
            If a name referenced in a formula is missing from both
            `params` and `covariates`.
        """
        namespace = dict(params)
        if covariates:
            namespace.update(covariates)

        missing = self.required_names - namespace.keys()
        if missing:
            raise KeyError(
                f"Formula references name(s) not found in params/covariates: {sorted(missing)}"
            )

        return {
            target: self._eval(tree.body, namespace)
            for target, tree in zip(self.targets, self.trees)
        }

    def _eval(self, node: ast.AST, namespace: Mapping[str, np.ndarray]) -> np.ndarray:
        if isinstance(node, ast.BinOp):
            left = self._eval(node.left, namespace)
            right = self._eval(node.right, namespace)
            return _BIN_OPS[type(node.op)](left, right)
        if isinstance(node, ast.UnaryOp):
            return _UNARY_OPS[type(node.op)](self._eval(node.operand, namespace))
        if isinstance(node, ast.Name):
            return namespace[node.id]
        if isinstance(node, ast.Constant):
            if not isinstance(node.value, (int, float)):
                raise ValueError(f"Only numeric constants allowed, got {node.value!r}")
            return node.value
        raise ValueError(f"Unsupported expression element: {ast.dump(node)}")

    def _validate(self, tree: ast.Expression, formula: str) -> None:
        for node in ast.walk(tree):
            if isinstance(node, ast.BinOp) and type(node.op) not in _BIN_OPS:
                raise ValueError(f"Unsupported operator in formula {formula!r}")
            if isinstance(node, ast.UnaryOp) and type(node.op) not in _UNARY_OPS:
                raise ValueError(f"Unsupported unary operator in formula {formula!r}")
            if isinstance(node, (ast.Call, ast.Attribute, ast.Subscript, ast.Lambda)):
                raise ValueError(
                    f"Calls/attributes/subscripts not allowed in formula {formula!r}"
                )

    @staticmethod
    def _collect_names(tree: ast.Expression) -> set[str]:
        return {n.id for n in ast.walk(tree) if isinstance(n, ast.Name)}

In [ ]:
import numpy as np

num_steps = 10
rng = np.random.default_rng(0)

# Define the design matrix
design = DesignMatrix([
    "v = v_0 + b_v * covariate",
    "tau = tau_0 + b_tau * n_cues",
])

# Randomly generated sampled parameters, each (num_steps,)
params = {
    "v_0":   rng.full(1.0, 0.1, size=num_steps),
    "b_v":   rng.normal(2.0, 0.2, size=num_steps),
    "tau_0": rng.normal(0.3, 0.02, size=num_steps),
    "b_tau": rng.normal(0.15, 0.01, size=num_steps),
    "a":     rng.normal(1.5, 0.1, size=num_steps),   # passthrough, untouched
}

# Randomly generated covariates, each (num_steps,)
covariates = {
    "covariate": rng.uniform(0.0, 1.0, size=num_steps),          # e.g. cue validity
    "n_cues":    rng.integers(1, 4, size=num_steps).astype(float),  # e.g. 1-3 cues
}

resolved = design.resolve(params, covariates)

# Merge passthrough params with resolved targets for the simulator
simulator_input = {**params, **resolved}

for key, arr in resolved.items():
    print(f"{key}: {np.round(arr, 3)}")

v: [2.489 1.468 2.409 1.125 1.535 1.315 1.982 2.637 1.41  0.988]
tau: [0.577 0.623 0.732 0.462 0.775 0.455 0.572 0.579 0.764 0.469]
